# Raster2Seq on our floor plans**What this answers.** Our stage 5 reads a plan by finding walls and flooding the spacebetween them. Raster2Seq skips that: it predicts **labelled room polygons directly** fromthe raster image — kitchen, living room, bedroom, bath, entry, storage — plus every doorand window as its own instance. If it works on estate-agent plans it replaces thewatershed, the caption seeding *and* the open-plan splitting problem in one move.It reports 88.7 room F1 on CubiCasa5K, MIT licence, published checkpoint. The only reasonwe have not already tried it is that it needs a GPU and two CUDA extensions compiled fromsource. That is what this notebook is for.**Before you start:** Runtime → Change runtime type → **T4 GPU**. Free tier is fine.**What you get back:** a zip containing, for every one of our 25 plans, the predicted roompolygons in our own pixel coordinates, a side-by-side picture, and the sameoutline-on-wall score the current engine is measured with — so the comparison islike-for-like rather than vibes.Roughly 25 minutes, most of it compiling.

## 1. Check the GPUIf this says "no GPU", stop and change the runtime type — everything below needs one.

In [ ]:
import subprocess, sysout = subprocess.run(["nvidia-smi"], capture_output=True, text=True)print(out.stdout or "NO GPU — Runtime > Change runtime type > T4 GPU, then rerun")import torchprint("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())assert torch.cuda.is_available(), "This notebook needs a GPU runtime."

## 2. Get the codeRaster2Seq vendors its own copy of detectron2, so there is no separate detectron2 installto fight with.

In [ ]:
%cd /content!git clone --depth 1 https://github.com/Cornell-VAILab/Raster2Seq.git 2>/dev/null || echo "already cloned"%cd /content/Raster2Seq!ls

## 3. Dependencies**This is the cell most likely to need a nudge.** `requirements.txt` pins versions from theauthors' 2024 environment — `scipy==1.8.1` and `scikit-image==0.19.0` predate Colab'sPython and will not build. So we install what inference actually needs and let the restfloat. If a later cell dies on a missing module, add it here rather than unpinningeverything.We deliberately do **not** touch the pre-installed torch: Colab's build matches its CUDAdriver, and replacing it is the fastest way to break the extension compile in §4.

In [ ]:
%pip install -q \    "numpy<2" opencv-python-headless timm shapely pycocotools huggingface_hub \    omegaconf fvcore cloudpickle "matplotlib" "scikit-image" "scipy" \    imageio descartes webcolors drawsvg svgpathtools svgwrite plyfile tqdmprint("done — a restart prompt from pip is safe to ignore unless a later cell complains")

## 4. Compile the two CUDA extensionsDeformable attention (from Deformable-DETR) and the differentiable rasteriser. Five to tenminutes, and it is nearly all of the notebook's setup time.`TORCH_CUDA_ARCH_LIST` is set explicitly because the build machine and the runtime GPU arethe same here, and letting the toolchain guess sometimes produces a binary for the wrongarchitecture that then fails at import with a confusing symbol error.

In [ ]:
import os, torchmajor, minor = torch.cuda.get_device_capability()os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"print("building for compute capability", os.environ["TORCH_CUDA_ARCH_LIST"],      "-", torch.cuda.get_device_name(0))%cd /content/Raster2Seq/models/ops!sh make.sh 2>&1 | tail -5%cd /content/Raster2Seq/diff_ras!python setup.py build develop 2>&1 | tail -5%cd /content/Raster2Seq

In [ ]:
# Both extensions must import cleanly before anything else is worth running.import sys; sys.path.insert(0, "/content/Raster2Seq")import MultiScaleDeformableAttention  # noqa: F401print("deformable attention: ok")try:    import diff_ras  # noqa: F401    print("differentiable rasteriser: ok")except ImportError as e:    print("differentiable rasteriser did not import:", e)    print("  Not fatal — it is only used by polygon refinement, which we disable below.")

## 5. Our plansPulled straight from the golden set's own recorded URLs, so nothing needs uploading andnothing needs to exist on your machine. 25 small PNGs.

In [ ]:
%cd /content!git clone --depth 1 https://github.com/romainbigare/visit-it.git 2>/dev/null || echo "already cloned"import json, urllib.request, sslfrom pathlib import PathPLANS = Path("/content/plans/raw"); PLANS.mkdir(parents=True, exist_ok=True)golden = json.loads(Path("/content/visit-it/data/golden/golden_set.json").read_text())ctx = ssl.create_default_context()got, missing = [], []for listing in golden["listings"]:    plans = listing.get("floorplans") or []    if not plans:        missing.append(listing["listing_id"]); continue    dest = PLANS / f"{listing['listing_id']}.png"    if not dest.exists():        try:            req = urllib.request.Request(plans[0]["url"], headers={"User-Agent": "Mozilla/5.0"})            dest.write_bytes(urllib.request.urlopen(req, context=ctx, timeout=60).read())        except Exception as exc:            print("  failed", listing["listing_id"], exc); missing.append(listing["listing_id"]); continue    got.append(listing["listing_id"])print(f"{len(got)} plans downloaded, {len(missing)} listings have no plan: {missing}")

## 6. Three versions of each planRaster2Seq was trained on CubiCasa5K, which is dark ink on white paper with no estate-agentcaptions. Our plans are neither, and we already know from the wall model that the mismatchmatters: on one tinted plan it labelled 62% of the flat "window" until the page waslevelled to white.So we prepare three variants and run all three. If **raw** wins, the model is robust and wecan drop the preprocessing; if **clean** wins, the preprocessing is load-bearing and belongsin the pipeline. Either answer is worth having.| variant | what it is ||---|---|| `raw` | the plan exactly as the agent published it || `white` | page levelled to white, darkest ink to black || `clean` | levelled, and every OCR word box painted out |

In [ ]:
%pip install -q pytesseract!apt-get -qq install -y tesseract-ocr > /dev/nullimport cv2, numpy as np, pytesseractfrom pathlib import Pathfrom PIL import ImageRAW = Path("/content/plans/raw")VARIANTS = {name: Path(f"/content/plans/{name}") for name in ("white", "clean")}for d in VARIANTS.values():    d.mkdir(parents=True, exist_ok=True)def load_rgb(path):    '''Composite transparency onto white -- several agent plans are LA-mode PNGs    whose ink lives entirely in the alpha channel, and a naive convert to RGB    turns them solid black.'''    im = Image.open(path)    if im.mode in ("LA", "RGBA", "P"):        im = im.convert("RGBA")        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))        im = Image.alpha_composite(bg, im)    return np.array(im.convert("RGB"))def whiten(rgb):    lum = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)    page = int(np.argmax(np.bincount(lum.ravel(), minlength=256)))    if page < 110:                      # light ink on a dark page        lum, page = 255 - lum, 255 - page    floor = float(np.percentile(lum, 1))    if page - floor < 20:        return cv2.cvtColor(lum, cv2.COLOR_GRAY2RGB)    out = np.clip((lum.astype(np.float32) - floor) * (255.0 / (page - floor)), 0, 255)    return cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_GRAY2RGB)def blank_text(rgb):    data = pytesseract.image_to_data(rgb, output_type=pytesseract.Output.DICT)    out = rgb.copy()    h, w = rgb.shape[:2]    for i, conf in enumerate(data["conf"]):        if float(conf) < 40 or not data["text"][i].strip():            continue        x, y = data["left"][i] - 2, data["top"][i] - 2        out[max(0, y):min(h, y + data["height"][i] + 4),            max(0, x):min(w, x + data["width"][i] + 4)] = 255    return outfor src in sorted(RAW.glob("*.png")):    rgb = load_rgb(src)    Image.fromarray(rgb).save(src)                     # normalise the raw copy too    w = whiten(rgb)    Image.fromarray(w).save(VARIANTS["white"] / src.name)    Image.fromarray(blank_text(w)).save(VARIANTS["clean"] / src.name)print("prepared", len(list(RAW.glob('*.png'))), "plans in 3 variants")

## 7. Predict`predict.py` walks a directory for images, so each variant is just a directory. The flagsare the authors' own `tools/predict_cc5k.sh` verbatim except for the paths — deliberately,so a bad result here is the model's and not our misconfiguration.`--disable_poly_refine` matches their published CubiCasa5K numbers, and also means thedifferentiable rasteriser is not needed even if it failed to build in §4.

In [ ]:
import subprocess, timeFLAGS = [    "--dataset_name=cubicasa", "--checkpoint=hf:cubicasa5k",    "--semantic_classes=12", "--input_channels", "3",    "--poly2seq", "--seq_len", "512", "--num_bins", "32",    "--disable_poly_refine", "--dec_attn_concat_src",    "--per_token_sem_loss", "--use_anchor", "--ema4eval", "--save_pred",]%cd /content/Raster2Seqfor variant in ("raw", "white", "clean"):    t0 = time.time()    cmd = ["python", "predict.py", f"--dataset_root=/content/plans/{variant}",           f"--output_dir=/content/preds/{variant}"] + FLAGS    r = subprocess.run(cmd, capture_output=True, text=True)    tail = (r.stdout + r.stderr).strip().splitlines()[-6:]    print(f"--- {variant}: exit {r.returncode} in {time.time()-t0:.0f}s")    for line in tail:        print("   ", line)

## 8. Put the polygons back in our coordinatesThe model works on a 256×256 letterboxed copy, so the polygons come out in *that* space.Undoing it is exact — resize by `min(256/h, 256/w)`, centre-pad, so invert in that order.This matters more than it sounds: the last time a coordinate space was assumed rather thaninverted, every outline in the review images was off by a couple of percent and correctrooms looked like they floated clear of their walls.

In [ ]:
import jsonfrom pathlib import Pathimport numpy as npfrom PIL import ImageCC5K_LABEL = {0: "Outdoor", 1: "Kitchen", 2: "Living Room", 3: "Bed Room", 4: "Bath",              5: "Entry", 6: "Storage", 7: "Garage", 8: "Undefined", 9: "Window", 10: "Door"}ROOM_CLASSES = set(range(0, 9))     # 9 and 10 are window and door instancesSIZE = 256def to_source_pixels(poly, src_w, src_h, size=SIZE):    '''Invert ResizeAndPad: undo the centre pad, then the aspect-preserving resize.'''    scale = min(size / src_h, size / src_w)    new_h, new_w = int(src_h * scale), int(src_w * scale)    left, top = (size - new_w) // 2, (size - new_h) // 2    p = np.asarray(poly, dtype=float).reshape(-1, 2)    return np.stack([(p[:, 0] - left) / scale, (p[:, 1] - top) / scale], axis=1)def collect(variant):    root = Path(f"/content/preds/{variant}")    jsons = sorted(root.rglob("jsons/*.json"))    out = {}    for jf in jsons:        lid = jf.stem        src = Image.open(f"/content/plans/{variant}/{lid}.png")        rooms, apertures = [], []        for inst in json.loads(jf.read_text()):            poly = to_source_pixels(inst["segmentation"], src.width, src.height)            rec = {"category_id": inst["category_id"],                   "label": CC5K_LABEL.get(inst["category_id"], "?"),                   "polygon_px": poly.tolist()}            (rooms if inst["category_id"] in ROOM_CLASSES else apertures).append(rec)        out[lid] = {"listing_id": lid, "image_size_px": [src.width, src.height],                    "rooms": rooms, "apertures": apertures}    return outresults = {v: collect(v) for v in ("raw", "white", "clean")}for v, r in results.items():    n_plans = len(r)    n_rooms = sum(len(x["rooms"]) for x in r.values())    n_ap = sum(len(x["apertures"]) for x in r.values())    per = f"{n_rooms / n_plans:.1f}" if n_plans else "-"    print(f"{v:6s}  {n_plans:2d} plans  {n_rooms:3d} rooms ({per} each)  {n_ap:3d} doors+windows")

## 9. Score it the same way we score ourselvesThe metric from `docs/PLAN-READING-REPORT.md`: walk each room outline and ask how much of itlies on a **wall** — using the CubiCasa wall segmenter as the wall reference, becausescoring against "any drawn line" cannot tell a room that traces its walls from one thattraces the kitchen cabinets.Two caveats worth keeping in view while you read the table. Raster2Seq predicts polygons at256×256, so its outlines are coarser than ours by construction and it is mildly penalisedon a fit metric measured at full resolution. And the wall reference is itself a prediction,so this favours nobody in particular but is not ground truth. **The pictures in §10 are thereal verdict.**

In [ ]:
%pip install -q segmentation-models-pytorch safetensorsimport syssys.path.insert(0, "/content/visit-it")from pathlib import Pathimport numpy as npfrom pipeline.floorplan import wallnetPath("/content/visit-it/models").mkdir(exist_ok=True)wallnet.MODEL_PATH = Path("/content/visit-it/models/plan_walls.safetensors")if not wallnet.MODEL_PATH.exists():    import urllib.request    urllib.request.urlretrieve(wallnet.MODEL_URL, wallnet.MODEL_PATH)print("wall reference available:", wallnet.available())

In [ ]:
import cv2def wall_reference(rgb):    '''The same barrier stage 5 uses, so the score means the same thing.'''    lum = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)    page = int(np.argmax(np.bincount(lum.ravel(), minlength=256)))    ink = (lum < max(30, page - 40)).astype(np.uint8)    return wallnet.barrier(rgb, ink, [])refs, scores = {}, {}for lid in sorted(results["raw"]):    rgb = np.array(Image.open(f"/content/plans/raw/{lid}.png").convert("RGB"))    refs[lid] = wall_reference(rgb)for variant, byid in results.items():    per_plan = {}    for lid, rec in byid.items():        ref = refs.get(lid)        if ref is None or not ref.any():            continue        polys = [r["polygon_px"] for r in rec["rooms"] if len(r["polygon_px"]) >= 3]        s = wallnet.outline_on_wall(polys, ref)        if s:            per_plan[lid] = s    scores[variant] = per_plan    flat = [v for s in per_plan.values() for v in s]    if flat:        print(f"{variant:6s}  {len(flat):3d} rooms  median outline-on-wall {np.median(flat):.3f}"              f"   >=0.8 {sum(1 for v in flat if v >= .8) / len(flat):5.1%}")print()print("current pipeline, same metric, same plans:  median 0.827   >=0.8 55.9%")

## 10. Look at themScore tables have already misled us once on this exact question. Read the pictures.For each plan: the plan on the left, Raster2Seq's rooms shaded and named on the right. Whatto look for specifically —- does an open-plan **"RECEPTION / DINING ROOM" come out as one room** (ours splits it)- do outlines **stop at kitchen cabinets** or run to the wall behind them- are **door swing arcs** treated as boundaries- are **bathrooms, WCs and hallways** found at all — ours often misses them- are the **room types** right, without needing the caption text

In [ ]:
import colorsysimport matplotlib.pyplot as pltfrom matplotlib.patches import Polygon as MplPolyBEST = "clean"   # change to "raw" or "white" once §9 says which one wondef hue(i):    r, g, b = colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.55, 0.95)    return (r, g, b)byid = results[BEST]ids = sorted(byid)for lid in ids:    rec = byid[lid]    rgb = np.array(Image.open(f"/content/plans/raw/{lid}.png").convert("RGB"))    fig, axes = plt.subplots(1, 2, figsize=(15, 8))    for ax in axes:        ax.imshow(rgb); ax.axis("off")    for i, room in enumerate(rec["rooms"]):        p = np.asarray(room["polygon_px"])        axes[1].add_patch(MplPoly(p, closed=True, facecolor=hue(i) + (0.35,),                                  edgecolor=hue(i), linewidth=2))        axes[1].text(*p.mean(axis=0), room["label"], ha="center", va="center",                     fontsize=9, weight="bold",                     bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))    for ap in rec["apertures"]:        p = np.asarray(ap["polygon_px"])        axes[1].add_patch(MplPoly(p, closed=True, facecolor="none",                                  edgecolor="crimson" if ap["label"] == "Door" else "royalblue",                                  linewidth=2))    med = np.median(scores[BEST].get(lid, [0]))    axes[0].set_title(f"{lid} — the plan", fontsize=11)    axes[1].set_title(f"Raster2Seq · {len(rec['rooms'])} rooms · "                      f"{len(rec['apertures'])} doors/windows · outline-on-wall {med:.2f}",                      fontsize=11)    plt.tight_layout(); plt.show()

## 11. Take the results homeA zip with the polygons in our own pixel coordinates and the scores, so the comparison canbe rerun and, if the answer is yes, wired into stage 5 as another engine behind the same`AD-4` interface — no artifact contract has to move.

In [ ]:
import json, shutilfrom pathlib import PathOUT = Path("/content/raster2seq_results"); OUT.mkdir(exist_ok=True)for variant, byid in results.items():    (OUT / f"{variant}.json").write_text(json.dumps(byid, indent=1))summary = {}for variant, per_plan in scores.items():    flat = [v for s in per_plan.values() for v in s]    summary[variant] = {        "rooms": len(flat),        "median_outline_on_wall": float(np.median(flat)) if flat else None,        "fraction_at_or_above_0.8": (sum(1 for v in flat if v >= .8) / len(flat)) if flat else None,        "per_listing_median": {k: float(np.median(v)) for k, v in per_plan.items()},    }summary["baseline_current_pipeline"] = {"median_outline_on_wall": 0.827,                                        "fraction_at_or_above_0.8": 0.559}(OUT / "summary.json").write_text(json.dumps(summary, indent=1))shutil.make_archive("/content/raster2seq_results", "zip", OUT)from google.colab import filesfiles.download("/content/raster2seq_results.zip")

## What the answer means**If the pictures look right** — open-plan spaces intact, outlines on walls not cabinets,bathrooms and hallways found, room types correct — then Raster2Seq becomes stage 5'sprimary engine and the watershed drops to a fallback. That also removes the need tofine-tune the wall model at all, because there would no longer be a wall model in thecritical path. Drop the zip in the repo and say so.**If they look wrong in a consistent way** — say every room is there but the polygons aretoo coarse at 256×256 — that is a resolution problem, not a capability problem, and theauthors publish a 512-resolution `Raster2Graph-512` checkpoint worth trying next.**If they look wrong in an inconsistent way**, the domain gap is real and the answer is thesame one as for the wall model: fine-tune on our own plans. That is`notebooks/finetune_wallnet_colab.ipynb`.